# 03 · Model Training — Rice Yield Prediction
### Train and compare Random Forest, XGBoost, and Gradient Boosting

**Author:** Ibrahim Denis Fofanah — Pace University | RiseAfrica Foundation for STEM and Innovation
**Project:** Machine Learning Approaches to Crop Yield Prediction and Post-Harvest Loss Reduction — Evidence from Sierra Leone

---

In [1]:
# ── Setup & project-root anchoring ────────────────────────────────────────────
import sys, os, warnings

# Resolve paths whether launched from repo root or from notebooks/
if not os.path.exists('data/raw') and os.path.exists('../data/raw'):
    os.chdir('..')
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 50)

for d in ['data/processed', 'outputs/figures', 'outputs/models', 'outputs/reports']:
    os.makedirs(d, exist_ok=True)

print(f'Working directory: {os.getcwd()}')

Working directory: /Users/ibrahimfofanah/sierraleone-agri-ml


## 1. Load Processed Dataset & Prepare Features

In [2]:
from src import feature_engineering as fe

ml_df = pd.read_csv('data/processed/processed_dataset.csv')

TARGET = 'Rice_Yield'
X, y, feature_names = fe.prepare_features(ml_df, target_col=TARGET)
print(f'Target mean: {y.mean():.0f} kg/ha')

[✓] Features: 151 | Samples: 25
    Target: Rice_Yield | Range: 785.6 – 2109.8
Target mean: 1441 kg/ha


## 2. Train/Test Split & Model Training

In [3]:
from sklearn.model_selection import train_test_split
from src import models as mdl

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

trained = mdl.train_all_models(X_train, y_train)

Train: 17 | Test: 8

Training models...
  ✓ Random Forest trained


  ✓ XGBoost trained
  ✓ GBM trained


## 3. Evaluate on Test Set

In [4]:
from src import evaluation as ev

results_df, predictions = ev.evaluate_models(trained, X_test, y_test)
results_df.to_csv('outputs/reports/model_performance.csv', index=False)
print(f'\nBest model: {results_df.iloc[0]["Model"]} (R² = {results_df.iloc[0]["R²"]:.4f})')
results_df


=== MODEL PERFORMANCE ON TEST SET ===
Model                        R²       RMSE        MAE
-------------------------------------------------------
Random Forest            0.8367     134.98     119.73
XGBoost                  0.9590      67.66      49.64
GBM                      0.9587      67.91      47.06

Best model: XGBoost (R² = 0.9590)


,Model,R²,RMSE,MAE
0,XGBoost,0.96,67.66,49.64
1,GBM,0.96,67.91,47.06
2,Random Forest,0.84,134.98,119.73


## 4. 5-Fold Cross-Validation

In [5]:
cv_df = ev.cross_validate_models(trained, X, y, cv=5)
cv_df.to_csv('outputs/reports/cross_validation_results.csv', index=False)
cv_df


=== 5-FOLD CROSS-VALIDATION ===
Model                     Mean R²     Std R²
---------------------------------------------


Random Forest              0.8223     0.1049


XGBoost                    0.8379     0.1559


GBM                        0.9223     0.0625


,Model,CV_R2_Mean,CV_R2_Std,CV_R2_Min,CV_R2_Max
0,GBM,0.92,0.06,0.80,0.98
1,XGBoost,0.84,0.16,0.60,0.98
2,Random Forest,0.82,0.10,0.68,0.92


## 5. Persist Trained Models

In [6]:
for name, model in trained.items():
    mdl.save_model(model, name, 'outputs/models')

# Save feature importance for the best model (used by notebooks 04 & 05)
best_name = results_df.iloc[0]['Model']
fi_df = ev.get_feature_importance(trained[best_name], feature_names, best_name)
fi_df.to_csv('outputs/reports/feature_importance.csv', index=False)
print(f'\n[saved] feature importance for {best_name}')
fi_df.head(10)

[✓] Model saved: outputs/models/Random_Forest.pkl
[✓] Model saved: outputs/models/XGBoost.pkl
[✓] Model saved: outputs/models/GBM.pkl

[saved] feature importance for XGBoost


,Feature,Importance,Model,Importance_Pct
0,Sweet Potato_Production,0.19,XGBoost,19.28
1,"Cereals, primary_Production",0.15,XGBoost,15.25
2,"Cereals, primary_Yield",0.13,XGBoost,13.11
3,Millet_Production,0.11,XGBoost,10.78
4,Rice_Yield_Lag1,0.09,XGBoost,8.96
5,Groundnuts_Yield,0.06,XGBoost,5.98
6,Rice_Prod_Per_Ha,0.06,XGBoost,5.96
7,Kola nuts_Yield,0.05,XGBoost,4.72
8,Fruit Primary_Yield,0.04,XGBoost,3.71
9,"Broad beans and horse beans, dry_Production",0.03,XGBoost,2.72
